# Data Preprocessing Pipeline - Patient No-Show Prediction
This notebook details the end-to-end data preprocessing steps for the Patient No-Show dataset, following the CRISP-ML(Q) methodology.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
We load the dataset and inspect the initial schema.

In [ ]:
df = pd.read_csv('./Dataset/noshowappointments-kagglev2-may-2016.csv')
print(f"Initial Shape: {df.shape}")
df.head()

## 2. Data Cleaning
Removing abnormal values (e.g., negative ages) and converting date strings to datetime objects.

In [ ]:
# Remove rows where Age is negative
df = df[df['Age'] >= 0]

# Convert dates
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
print("Date conversion and age filtering complete.")

## 3. Feature Engineering
Creating informative features like `WaitingTime`, `DayOfWeek`, and `Month`.

In [ ]:
# Calculate Waiting Time in days
df['WaitingTime'] = (df['AppointmentDay'].dt.normalize() - df['ScheduledDay'].dt.normalize()).dt.days
df['WaitingTime'] = df['WaitingTime'].apply(lambda x: max(x, 0))

# Extract Temporal features
df['AppointmentDayOfWeek'] = df['AppointmentDay'].dt.dayofweek
df['AppointmentMonth'] = df['AppointmentDay'].dt.month

# Mapping Target
df['Target'] = df['No-show'].map({'No': 0, 'Yes': 1})

# Dropping redundant/leakage columns
cols_to_drop = ['PatientId', 'AppointmentID', 'ScheduledDay', 'AppointmentDay', 'No-show']
df_final = df.drop(columns=cols_to_drop)
df_final.head()

## 4. Visual Analysis (EDA)
Analyzing distributions before transformation.

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x='Target', data=df_final, palette='viridis')
plt.title('Target Distribution (0: Show, 1: No-Show)')
plt.show()

## 5. Data Transformation
Defining numeric and categorical pipelines using `ColumnTransformer`.

In [ ]:
categorical_features = ['Gender', 'Neighbourhood']
numeric_features = ['Age', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'WaitingTime']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X = df_final.drop(columns=['Target'])
y = df_final['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_processed = preprocessor.fit_transform(X_train)
print(f"Processed Data Shape: {X_train_processed.shape}")